<a href="https://colab.research.google.com/github/jrlewis-umbc/MPLNET/blob/python/mplnet_data_plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*   **Module:** mplnet_data_plots.ipynb
*   **Purpose:** To read MPLNET data utilizing the Application Programming Interface (API) and make curtain plots and line plots
*   **Author(s):** Sophia Summers, Jasper Lewis







Import necessary modules


In [9]:
import requests
import io
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime, timedelta
from google.colab import files

Customize inputs

In [51]:
version = "V3"                # Enter V2 or V3
level = "L1"                  # Enter L1, L15, or L2 (if available)
product = "NRB"               # Enter NRB, CLD, AER, or PBL
start_date_str = "20240728"   # Enter in YYYYMMDD format
end_date_str = "20240729"     # Enter in YYYYMMDD format
site = "GSFC"           # Site name
var = "nrb"       # Variable to plot (For full list of variables, see Product Info Page: mplnet.gsfc.nasa.gov/product-info/)

confidence = None             # "high", "moderate", "low", or None
day_or_night = None       # "day", "night", or None if you want to look at all times

# Change the following parameters ymin and ymax to set minimum and maximum altitudes to be plotted for 2D variables or range of values
# of interest for 1D variables. Parameters zmin and zmax are for 2D curtain plots.
ymin = 0.0
ymax = 10.0
zmin = 0.0
zmax = 3.0

# Ensure the inputs match the expected format
start_date_str = start_date_str.replace("-","").replace("/","").replace(" ","")
end_date_str = end_date_str.replace("-","").replace("/","").replace(" ","")
level = str(level).replace(".","")
product = product.upper()
var = var.lower()
qa_var = "qa_" + var
if confidence is not None:
  confidence = confidence.lower()
if day_or_night is not None:
  day_or_night = day_or_night.lower()

# Determine datetime from string
start_date = datetime.strptime(start_date_str, "%Y%m%d")
end_date = datetime.strptime(end_date_str, "%Y%m%d")
delta = timedelta(days=1)



Obtain data using MPLNET Web Services API. If requested screen day/night retrievals. ***Warning: Selecting time frames longer than two weeks can cause lengthy download times***

In [52]:
datasets = []

current_date = start_date

if day_or_night is not None:
  while current_date <= end_date:
      yyyy = current_date.strftime("%Y")
      mm = current_date.strftime("%m")
      dd = current_date.strftime("%d")
      data_url = 'https://mplnet.gsfc.nasa.gov/download?version='+str(version)+'&level='+str(level)+'&product='+product+'&site='+site+"&year="+yyyy+"&month="+mm+"&day="+dd
      day_url = 'https://mplnet.gsfc.nasa.gov/download?version='+str(version)+'&level='+str(level)+'&product=CLD&site='+site+"&year="+yyyy+"&month="+mm+"&day="+dd

      response = requests.get(data_url)
      response.raise_for_status()   #confirms that the URL exists

      day_response = requests.get(day_url)   #retrieves CLD product data
      day_response.raise_for_status()

      try:
        file_bytes = io.BytesIO(response.content)
        ds = xr.open_dataset(file_bytes, engine = "h5netcdf", decode_times = False) #opens a dataset
        #Converting time units
        ds = ds.assign_coords(time=pd.to_datetime(ds["time"].values, unit="D", origin="julian"))

        day_file_bytes = io.BytesIO(day_response.content)
        day_ds = xr.open_dataset(day_file_bytes, engine = "h5netcdf", decode_times = False) #opens CLD product dataset
        #Converting time units for CLD dataset (using its own time variable)
        day_ds = day_ds.assign_coords(time=pd.to_datetime(day_ds["time"].values, unit="D", origin="julian"))

        #Apply screening based on day_indicator from CLD product
        if "day_indicator" in day_ds:
          #Create a mask using the day_indicator variable
          if day_or_night == "day":
            day_mask = day_ds["day_indicator"] == 2
          elif day_or_night == "night":
            day_mask = day_ds["day_indicator"] == 1
          #Apply the mask
          ds = ds.where(day_mask)
        else:
            print(f"Warning: 'day_indicator' not found in CLD product for {current_date.strftime('%Y-%m-%d')}. No day/night screening applied.")

        datasets.append(ds)

      except Exception:
          current_date += delta
          continue

      current_date += delta

else:
  while current_date <= end_date:
      yyyy = current_date.strftime("%Y")
      mm = current_date.strftime("%m")
      dd = current_date.strftime("%d")
      data_url = 'https://mplnet.gsfc.nasa.gov/download?version='+str(version)+'&level='+str(level)+'&product='+product+'&site='+site+"&year="+yyyy+"&month="+mm+"&day="+dd

      response = requests.get(data_url)   #retrieves data from the URL
      response.raise_for_status()   #confirms that the URL exists

      try:
        file_bytes = io.BytesIO(response.content)
        ds = xr.open_dataset(file_bytes, engine = "h5netcdf", decode_times = False) #opens a dataset
        #Converting time units
        ds = ds.assign_coords(time=pd.to_datetime(ds["time"].values, unit="D", origin="julian"))
        datasets.append(ds)

      except Exception:
          current_date += delta
          continue

      current_date += delta

Make plot of selected variable versus time. Screen data based on QA level, if requested.  

In [ ]:
combined = xr.concat(datasets, dim="time")
if combined[var].ndim == 1:
    var_to_plot = combined[var]
else:
    var_to_plot = combined[var].squeeze()

if qa_var in combined and confidence is not None:
  qa = combined[qa_var].squeeze()
  if confidence == "high":
    good = qa == 1
  elif confidence == "moderate":
    good = qa <= 2
  elif confidence == "low":
    good = qa <=4
  var_to_plot = var_to_plot.where(good) #applies a mask to the data so that we only look at data of the given confidence level

time = combined["time"]

plt.figure(figsize=(12, 6))
plot_title = site + " " + var.title() + " " + start_date_str

if start_date_str == end_date_str:
  if day_or_night is None:
    plt.title(plot_title)
  else:
    plt.title(plot_title + " (" + day_or_night.title() + ")")
else:
  if day_or_night is None:
    plt.title(plot_title + " to " + end_date_str)
  else:
    plt.title(plot_title + " to " + end_date_str + " (" + day_or_night.title() + ")")

if var_to_plot.ndim == 1:

  plt.plot(time, var_to_plot, marker="o", markersize=4, linewidth=1)
  plt.xlabel("Time (UTC)")
  plt.ylabel(ds[var].attrs.get("unit", var).title())


else: #for 2D variables
  var_to_plot = var_to_plot.values
  altitude = combined["altitude"].values

  time_2d = np.broadcast_to(time.values[:, np.newaxis], altitude.shape)

  cmap = cm.get_cmap('turbo').copy()
  cmap.set_over('white')
  im = plt.pcolormesh(
      time_2d,      #X-axis: 2D array of times
      altitude,     #Y-axis: 2D array of altitudes
      var_to_plot,  #Z-data: 2D array of variable values
      vmin=zmin,
      vmax=zmax,
      cmap=cmap,
      shading = "auto"
  )

  cbtitle = combined[var].attrs.get('long_name', var).title()
  cbtitle += " ("+combined[var].attrs.get('units', var)+")"
  plt.colorbar(im, label=cbtitle)
  plt.xlabel("Time (UTC)")
  plt.ylabel("Altitude (km)")


plt.ylim(ymin, ymax)

ax = plt.gca()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
plt.xticks(rotation=0)

title = plt.gca().get_title()

plt.savefig(title+".png", bbox_inches="tight", dpi=300)

plt.show()

Save image to PNG file

In [ ]:
files.download(title+".png")